# Mammography Noise Filtering, U-Net ROI Masking, and DenseNet Classification

This notebook is organized as a full mammography workflow for Mini-DDSM:

1. Build a clean image manifest.
2. Inspect class balance and noisy samples.
3. Apply mammography-friendly filtering functions.
4. Train an optional U-Net to segment the breast/foreground region.
5. Train a DenseNet classifier instead of EfficientNet.
6. Evaluate the model and run predictions with Grad-CAM explanations.

The filtering stage is important because the sample image contains scanner labels, black borders, and visible noise. The pipeline uses ROI cropping, CLAHE contrast enhancement, median/bilateral/non-local means denoising, and optional unsharp masking.

## 1. Imports and Configuration

In [ ]:
import os
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_ROOT = Path("/kaggle/input/miniddsm/Mini_DDSM_Upload")
WORK_DIR = Path("/kaggle/working")

IMG_SIZE = 384
BATCH_SIZE = 8
NUM_CLASSES = 3
EPOCHS_CLASSIFIER_HEAD = 10
EPOCHS_CLASSIFIER_FINE = 25
EPOCHS_UNET = 8
AUTOTUNE = tf.data.AUTOTUNE

CLASS_FOLDERS = {
    "Normal": 0,
    "Benign": 1,
    "Cancer": 2,
}

CLASS_NAMES = ["Normal", "Benign", "Malignant"]
print("TensorFlow:", tf.__version__)

## 2. Dataset Manifest

In [ ]:
def build_manifest(data_root: Path, class_folders: dict) -> pd.DataFrame:
    rows = []
    for folder, label in class_folders.items():
        folder_path = data_root / folder
        image_paths = []
        for pattern in ("*.jpg", "*.jpeg", "*.png"):
            image_paths.extend(folder_path.rglob(pattern))

        for image_path in image_paths:
            rows.append({
                "filepath": str(image_path),
                "label": label,
                "class_name": "Malignant" if folder == "Cancer" else folder,
            })

    df = pd.DataFrame(rows)
    if df.empty:
        raise FileNotFoundError(f"No images found under {data_root}. Check DATA_ROOT.")
    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)


df = build_manifest(DATA_ROOT, CLASS_FOLDERS)
print(df.head())
print("\nClass distribution:")
print(df["class_name"].value_counts())

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=SEED,
)

class_weight_values = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=train_df["label"].values,
)
class_weights = {i: float(w) for i, w in enumerate(class_weight_values)}

# Increase malignant-class importance. Tune this after reviewing recall/precision.
class_weights[2] *= 1.5

print("Train:", train_df.shape, "Validation:", val_df.shape)
print("Class weights:", class_weights)

## 3. Noise Filtering and ROI Functions

In [ ]:
def read_grayscale_image(path: str) -> np.ndarray:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f"Could not read image: {path}")
    return image


def crop_breast_roi(image: np.ndarray, threshold: int = 8, pad: int = 12) -> np.ndarray:
    """Remove black borders and scanner background while preserving breast tissue."""
    if image.ndim != 2:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    mask = image > threshold
    if not np.any(mask):
        return image

    ys, xs = np.where(mask)
    y1, y2 = max(ys.min() - pad, 0), min(ys.max() + pad, image.shape[0] - 1)
    x1, x2 = max(xs.min() - pad, 0), min(xs.max() + pad, image.shape[1] - 1)
    return image[y1:y2 + 1, x1:x2 + 1]


def apply_clahe(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size=(8, 8)) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(image.astype(np.uint8))


def median_filter(image: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    return cv2.medianBlur(image.astype(np.uint8), kernel_size)


def gaussian_filter(image: np.ndarray, kernel_size: int = 5, sigma: float = 0) -> np.ndarray:
    return cv2.GaussianBlur(image.astype(np.uint8), (kernel_size, kernel_size), sigma)


def bilateral_filter(image: np.ndarray, diameter: int = 7, sigma_color: int = 45, sigma_space: int = 45) -> np.ndarray:
    return cv2.bilateralFilter(image.astype(np.uint8), diameter, sigma_color, sigma_space)


def non_local_means_filter(image: np.ndarray, h: int = 8) -> np.ndarray:
    return cv2.fastNlMeansDenoising(image.astype(np.uint8), None, h, 7, 21)


def unsharp_mask(image: np.ndarray, amount: float = 1.2, radius: int = 5) -> np.ndarray:
    blurred = cv2.GaussianBlur(image.astype(np.uint8), (radius, radius), 0)
    sharpened = cv2.addWeighted(image.astype(np.uint8), 1 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def mammogram_filter_pipeline(image: np.ndarray) -> np.ndarray:
    """Recommended default for noisy mammograms: crop, denoise, enhance contrast, lightly sharpen."""
    image = crop_breast_roi(image)
    image = median_filter(image, kernel_size=5)
    image = bilateral_filter(image, diameter=7, sigma_color=45, sigma_space=45)
    image = apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8))
    image = unsharp_mask(image, amount=0.6, radius=5)
    return image

In [ ]:
def compare_filters(image_path: str):
    image = read_grayscale_image(image_path)
    cropped = crop_breast_roi(image)

    variants = {
        "Original": image,
        "ROI crop": cropped,
        "Median": median_filter(cropped),
        "Gaussian": gaussian_filter(cropped),
        "Bilateral": bilateral_filter(cropped),
        "NLM": non_local_means_filter(cropped),
        "CLAHE": apply_clahe(cropped),
        "Recommended": mammogram_filter_pipeline(image),
    }

    plt.figure(figsize=(16, 8))
    for i, (title, img) in enumerate(variants.items(), start=1):
        plt.subplot(2, 4, i)
        plt.imshow(img, cmap="gray")
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


sample_path = train_df.sample(1, random_state=SEED)["filepath"].iloc[0]
compare_filters(sample_path)

## 4. TensorFlow Preprocessing

In [ ]:
def preprocess_mammogram_np(path_bytes) -> np.ndarray:
    path = path_bytes.decode("utf-8")
    image = read_grayscale_image(path)
    image = mammogram_filter_pipeline(image)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    return image.astype(np.float32)


def load_classifier_image(path, label):
    image = tf.numpy_function(preprocess_mammogram_np, [path], tf.float32)
    image.set_shape((IMG_SIZE, IMG_SIZE, 3))
    label = tf.one_hot(tf.cast(label, tf.int32), NUM_CLASSES)
    return image, label


data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.03),
        layers.RandomZoom(0.08),
        layers.RandomContrast(0.15),
    ],
    name="mammography_augmentation",
)


def make_classifier_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices((frame["filepath"].values, frame["label"].values))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_classifier_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_classifier_dataset(train_df, training=True)
val_ds = make_classifier_dataset(val_df, training=False)

## 5. Optional U-Net Breast ROI Segmentation

Mini-DDSM classification folders do not always include pixel-level lesion masks. The U-Net below is therefore set up to learn a foreground breast mask from pseudo-labels generated by thresholding and morphology. If you have real masks later, replace `make_pseudo_mask` with a mask loader.

The mask can help remove black background, labels, and scanner artifacts before classification or during inference.

In [ ]:
def make_pseudo_mask(image: np.ndarray) -> np.ndarray:
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    blurred = cv2.GaussianBlur(image.astype(np.uint8), (5, 5), 0)
    _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels > 1:
        largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        mask = np.where(labels == largest, 255, 0).astype(np.uint8)

    kernel = np.ones((9, 9), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return (mask > 0).astype(np.float32)


def preprocess_unet_np(path_bytes):
    path = path_bytes.decode("utf-8")
    image = read_grayscale_image(path)
    image = mammogram_filter_pipeline(image)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    mask = make_pseudo_mask(image)
    image = (image.astype(np.float32) / 255.0)[..., None]
    mask = mask[..., None].astype(np.float32)
    return image, mask


def load_unet_pair(path):
    image, mask = tf.numpy_function(preprocess_unet_np, [path], [tf.float32, tf.float32])
    image.set_shape((IMG_SIZE, IMG_SIZE, 1))
    mask.set_shape((IMG_SIZE, IMG_SIZE, 1))
    return image, mask


def make_unet_dataset(frame: pd.DataFrame, training: bool) -> tf.data.Dataset:
    ds = tf.data.Dataset.from_tensor_slices(frame["filepath"].values)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_unet_pair, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


unet_train_ds = make_unet_dataset(train_df, training=True)
unet_val_ds = make_unet_dataset(val_df, training=False)

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.Activation("relu")(x)


def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 1), base_filters=32):
    inputs = keras.Input(shape=input_shape)

    c1 = conv_block(inputs, base_filters)
    p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, base_filters * 2)
    p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, base_filters * 4)
    p3 = layers.MaxPooling2D()(c3)
    c4 = conv_block(p3, base_filters * 8)
    p4 = layers.MaxPooling2D()(c4)

    bottleneck = conv_block(p4, base_filters * 16)

    u4 = layers.UpSampling2D()(bottleneck)
    u4 = layers.Concatenate()([u4, c4])
    c5 = conv_block(u4, base_filters * 8)
    u3 = layers.UpSampling2D()(c5)
    u3 = layers.Concatenate()([u3, c3])
    c6 = conv_block(u3, base_filters * 4)
    u2 = layers.UpSampling2D()(c6)
    u2 = layers.Concatenate()([u2, c2])
    c7 = conv_block(u2, base_filters * 2)
    u1 = layers.UpSampling2D()(c7)
    u1 = layers.Concatenate()([u1, c1])
    c8 = conv_block(u1, base_filters)

    outputs = layers.Conv2D(1, 1, activation="sigmoid", name="mask")(c8)
    return keras.Model(inputs, outputs, name="unet_breast_roi")


def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1, 2, 3))
    union = tf.reduce_sum(y_true, axis=(1, 2, 3)) + tf.reduce_sum(y_pred, axis=(1, 2, 3))
    return tf.reduce_mean((2.0 * intersection + smooth) / (union + smooth))


def dice_bce_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + (1.0 - dice_coef(y_true, y_pred))


unet_model = build_unet()
unet_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=dice_bce_loss,
    metrics=[dice_coef, keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="foreground_iou")],
)
unet_model.summary()

In [ ]:
unet_callbacks = [
    keras.callbacks.ModelCheckpoint(
        WORK_DIR / "best_unet_breast_roi.keras",
        monitor="val_dice_coef",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_dice_coef",
        mode="max",
        patience=4,
        restore_best_weights=True,
        verbose=1,
    ),
]

unet_history = unet_model.fit(
    unet_train_ds,
    validation_data=unet_val_ds,
    epochs=EPOCHS_UNET,
    callbacks=unet_callbacks,
)

In [ ]:
def show_unet_masks(model, frame: pd.DataFrame, n: int = 3):
    paths = frame.sample(n, random_state=SEED)["filepath"].tolist()
    plt.figure(figsize=(12, 4 * n))
    for row, path in enumerate(paths):
        image, pseudo = preprocess_unet_np(path.encode("utf-8"))
        pred = model.predict(image[None, ...], verbose=0)[0, ..., 0]

        plt.subplot(n, 3, row * 3 + 1)
        plt.imshow(image[..., 0], cmap="gray")
        plt.title("Filtered image")
        plt.axis("off")

        plt.subplot(n, 3, row * 3 + 2)
        plt.imshow(pseudo[..., 0], cmap="gray")
        plt.title("Pseudo mask")
        plt.axis("off")

        plt.subplot(n, 3, row * 3 + 3)
        plt.imshow(pred > 0.5, cmap="gray")
        plt.title("U-Net prediction")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_unet_masks(unet_model, val_df, n=3)

## 6. DenseNet Classifier

In [ ]:
def build_densenet_classifier(img_size=IMG_SIZE, num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=(img_size, img_size, 3), name="image")

    # load_classifier_image returns RGB values in 0..255.
    x = keras.applications.densenet.preprocess_input(inputs)

    base_model = keras.applications.DenseNet121(
        include_top=False,
        weights="imagenet",
        input_tensor=x,
    )
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.45)(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probs")(x)

    model = keras.Model(inputs, outputs, name="densenet121_mammography_classifier")
    return model, base_model


densenet_model, densenet_base = build_densenet_classifier()
densenet_model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss=keras.losses.CategoricalFocalCrossentropy(gamma=2.0, alpha=[0.15, 0.30, 0.55]),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.AUC(name="auc", multi_label=True, num_labels=NUM_CLASSES),
        keras.metrics.Recall(class_id=2, name="malignant_recall"),
        keras.metrics.Precision(class_id=2, name="malignant_precision"),
    ],
)
densenet_model.summary()

In [ ]:
classifier_callbacks = [
    keras.callbacks.ModelCheckpoint(
        WORK_DIR / "best_densenet121_mammography.keras",
        monitor="val_malignant_recall",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_malignant_recall",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
]

history_head = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CLASSIFIER_HEAD,
    class_weight=class_weights,
    callbacks=classifier_callbacks,
)

In [ ]:
# Fine-tune the top DenseNet layers after the classification head has warmed up.
densenet_base.trainable = True
for layer in densenet_base.layers[:-80]:
    layer.trainable = False

densenet_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss=keras.losses.CategoricalFocalCrossentropy(gamma=2.0, alpha=[0.15, 0.30, 0.55]),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.AUC(name="auc", multi_label=True, num_labels=NUM_CLASSES),
        keras.metrics.Recall(class_id=2, name="malignant_recall"),
        keras.metrics.Precision(class_id=2, name="malignant_precision"),
    ],
)

history_fine = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CLASSIFIER_FINE,
    class_weight=class_weights,
    callbacks=classifier_callbacks,
)

## 7. Evaluation

In [ ]:
def collect_predictions(model, dataset):
    y_true, y_prob = [], []
    for images, labels in dataset:
        probs = model.predict(images, verbose=0)
        y_prob.append(probs)
        y_true.append(labels.numpy())
    y_true = np.vstack(y_true)
    y_prob = np.vstack(y_prob)
    return y_true, y_prob


y_true_oh, y_prob = collect_predictions(densenet_model, val_ds)
y_true = np.argmax(y_true_oh, axis=1)
y_pred = np.argmax(y_prob, axis=1)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap="Blues", values_format="d")
plt.title("DenseNet121 Validation Confusion Matrix")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
for i, class_name in enumerate(CLASS_NAMES):
    fpr, tpr, _ = roc_curve(y_true_oh[:, i], y_prob[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_name} AUC = {roc_auc:.2f}")

plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Validation ROC Curves")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

## 8. Prediction and Grad-CAM

In [ ]:
def prepare_single_image(image_path: str) -> np.ndarray:
    image = preprocess_mammogram_np(str(image_path).encode("utf-8"))
    return image[None, ...]


def predict_image(image_path: str, model=densenet_model):
    image = prepare_single_image(image_path)
    probs = model.predict(image, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    return CLASS_NAMES[pred_idx], float(probs[pred_idx]), probs


sample_image = val_df.sample(1, random_state=7)["filepath"].iloc[0]
pred_class, confidence, probs = predict_image(sample_image)
print("Image:", sample_image)
print(f"Prediction: {pred_class} ({confidence:.2%})")
print(dict(zip(CLASS_NAMES, probs.round(4))))

In [ ]:
def find_last_conv_layer(model: keras.Model) -> str:
    for layer in reversed(model.layers):
        if isinstance(layer, layers.Conv2D):
            return layer.name
        if isinstance(layer, keras.Model):
            for nested in reversed(layer.layers):
                if isinstance(nested, layers.Conv2D):
                    return nested.name
    raise ValueError("No Conv2D layer found for Grad-CAM.")


LAST_CONV_LAYER = find_last_conv_layer(densenet_base)
print("Grad-CAM layer:", LAST_CONV_LAYER)


def make_gradcam_heatmap(image_batch, classifier, conv_base, last_conv_layer_name, class_index=None):
    grad_model = keras.Model(
        inputs=classifier.input,
        outputs=[conv_base.get_layer(last_conv_layer_name).output, classifier.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_batch)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        class_score = predictions[:, class_index]

    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    return heatmap / (tf.reduce_max(heatmap) + 1e-8)


def show_gradcam(image_path: str, model=densenet_model, base=densenet_base):
    image_batch = prepare_single_image(image_path)
    pred_class, confidence, probs = predict_image(image_path, model)
    class_index = int(np.argmax(probs))

    heatmap = make_gradcam_heatmap(image_batch, model, base, LAST_CONV_LAYER, class_index).numpy()
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    display_img = np.clip(image_batch[0], 0, 255).astype(np.uint8)
    overlay = cv2.addWeighted(display_img, 0.62, heatmap, 0.38, 0)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(display_img)
    plt.title("Filtered mammogram")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(overlay)
    plt.title(f"Grad-CAM: {pred_class} ({confidence:.1%})")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


show_gradcam(sample_image)

## 9. Save Final Models

In [ ]:
densenet_model.save(WORK_DIR / "mammography_densenet121_final.keras")
unet_model.save(WORK_DIR / "mammography_unet_roi_final.keras")
print("Saved models to:", WORK_DIR)